In [ ]:
# pip install fairchem-core

from fairchem.core import pretrained_mlip
from ase.optimize import BFGS
from ase.io import read, write
from ase.build import molecule

device = "cpu"

# UMA-ODAC calculator
calc = pretrained_mlip.get_predict_unit(
    model_name="uma-odac",
    device=device
)

# Optimize bare MOF
mof = read("mg_mof74.cif")
mof.calc = calc
opt_mof = BFGS(mof, logfile="mof_opt.log")
opt_mof.run(fmax=0.01)

E_mof = mof.get_potential_energy()
print(f"E(Mg-MOF-74) = {E_mof:.6f} eV")

# Optimize isolated CO2
co2 = molecule("CO2")
co2.calc = calc
opt_co2 = BFGS(co2, logfile="co2_opt.log")
opt_co2.run(fmax=0.01)

E_co2 = co2.get_potential_energy()
print(f"E(CO2) = {E_co2:.6f} eV")

# Optimize MOF + CO2
mof_co2 = read("mof_co2_initial.cif")
mof_co2.calc = calc
opt = BFGS(mof_co2, logfile="mof_co2_opt.log")
opt.run(fmax=0.01)

E_mof_co2 = mof_co2.get_potential_energy()
print(f"E(CO2@Mg-MOF-74) = {E_mof_co2:.6f} eV")

write("mof_co2_relaxed.cif", mof_co2)

# Adsorption energy
delta_E = E_mof_co2 - (E_mof + E_co2)
print(f"Adsorption energy: {delta_E:.4f} eV")

# Mg–CO2 distance
mg_indices = [i for i, atom in enumerate(mof_co2) if atom.symbol == "Mg"]
co2_indices = list(range(len(mof_co2)-3, len(mof_co2)))

for mg_i in mg_indices:
    for co2_j in co2_indices:
        d = mof_co2.get_distance(mg_i, co2_j, mic=True)
        if d < 3.5:
            print(f"Mg[{mg_i}] — {mof_co2[co2_j].symbol}[{co2_j}]: {d:.3f} Å")

ModuleNotFoundError: No module named 'fairchem'